# 003: Multilayer Perceptron Architecture — Universal approximation, hidden layers, and XOR solution

## 1. THEORY LAYER
- **Multilayer Perceptron (MLP)**: A feedforward neural network with one or more hidden layers between input and output.
- **Universal Approximation Theorem (Cybenko 1989, Hornik 1991)**: A feedforward network with a single hidden layer containing a finite number of neurons can approximate any continuous function on a compact subset of $\mathbb{R}^n$ to arbitrary precision, given a suitable activation function.
- **Why Hidden Layers?**: They create non-linear decision boundaries by composing linear transformations with non-linear activations, enabling the network to solve problems like XOR.

## 2. MLP FORWARD EQUATIONS
- Layer $l$ computation:
  $$\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$
  $$\mathbf{a}^{[l]} = \sigma(\mathbf{z}^{[l]})$$
- The input layer is $\mathbf{a}^{[0]} = \mathbf{x}$.

---


In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class SimpleMLP:
    """2-layer MLP (1 hidden layer) solving XOR from scratch."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        np.random.seed(42)
        # Layer 1: input -> hidden
        self.W1 = np.random.randn(hidden_dim, input_dim) * 0.5
        self.b1 = np.zeros((hidden_dim, 1))
        # Layer 2: hidden -> output
        self.W2 = np.random.randn(output_dim, hidden_dim) * 0.5
        self.b2 = np.zeros((output_dim, 1))

    def forward(self, X):
        """Forward pass through 2 layers."""
        # X shape: (input_dim, m) where m = number of samples
        self.Z1 = self.W1 @ X + self.b1          # Hidden pre-activation
        self.A1 = sigmoid(self.Z1)                 # Hidden activation
        self.Z2 = self.W2 @ self.A1 + self.b2     # Output pre-activation
        self.A2 = sigmoid(self.Z2)                 # Output activation (prediction)
        return self.A2

    def compute_loss(self, Y, A2):
        """Binary cross-entropy loss."""
        m = Y.shape[1]
        loss = -(1/m) * np.sum(Y * np.log(A2 + 1e-8) + (1 - Y) * np.log(1 - A2 + 1e-8))
        return loss

    def backward(self, X, Y, lr=1.0):
        """Backpropagation and weight update."""
        m = Y.shape[1]
        # Output layer gradients
        dZ2 = self.A2 - Y                                     # (1, m)
        dW2 = (1/m) * dZ2 @ self.A1.T                        # (1, hidden)
        db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)     # (1, 1)
        # Hidden layer gradients (chain rule)
        dA1 = self.W2.T @ dZ2                                 # (hidden, m)
        dZ1 = dA1 * self.A1 * (1 - self.A1)                  # sigmoid derivative
        dW1 = (1/m) * dZ1 @ X.T                               # (hidden, input)
        db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)     # (hidden, 1)
        # Update parameters
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1



In [ ]:
# ── Train MLP on XOR ──
print("--- Training MLP on XOR ---")
X = np.array([[0,0],[0,1],[1,0],[1,1]]).T   # (2, 4)
Y = np.array([[0, 1, 1, 0]])                # (1, 4)

mlp = SimpleMLP(input_dim=2, hidden_dim=4, output_dim=1)

for epoch in range(5001):
    A2 = mlp.forward(X)
    loss = mlp.compute_loss(Y, A2)
    mlp.backward(X, Y, lr=2.0)
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss: {loss:.6f}")



In [ ]:
print("\n--- Final XOR Predictions ---")
predictions = mlp.forward(X)
for i in range(4):
    print(f"Input: [{X[0,i]}, {X[1,i]}] | Target: {Y[0,i]} | Predicted: {predictions[0,i]:.4f} | Class: {int(predictions[0,i] > 0.5)}")
